# Train-Ready Grammar Data EDA

This notebook focuses on the final cleaned dataset used for BabyLLaMA fine-tuning.

It is designed for files such as `../../data/processed/train_ready_grammar_data.jsonl`, and it uses the same official `count_word` method from `Language_acquisition.ipynb` to count words in `good` and `bad`.

In [11]:
import json
from pathlib import Path

import nltk
import pandas as pd
from nltk.tokenize import word_tokenize

In [12]:
for resource in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

In [13]:
TRAIN_READY_PATH = Path("../../data\processed\mixed_final_generated_grammar_data.jsonl")

In [14]:
def count_word(text):
    tokens = word_tokenize(text)
    return len(tokens)

In [15]:
def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [16]:
records = load_jsonl(TRAIN_READY_PATH)
df = pd.DataFrame(records)
print(f"Loaded {len(df)} train-ready records from {TRAIN_READY_PATH}")
df.head()

Loaded 48393 train-ready records from ..\..\data\processed\mixed_final_generated_grammar_data.jsonl


,id,phenomenon,topic,good,bad,edit_type,prompt_family,parent_llm
0,openai_prompt_3_c4e80b6851,anaphor_agreement,family_home,Leah reminded herself to lock the gate.,Leah reminded themselves to lock the gate.,reflexive_number_match,prompt_3,openai
1,openai_prompt_3_71c80d8303,anaphor_agreement,family_home,The brothers dressed themselves before dinner.,The brothers dressed himself before dinner.,reflexive_number_match,prompt_3,openai
2,openai_prompt_3_27aa320f3c,anaphor_agreement,family_home,Aunt Nora blamed herself for the spill.,Aunt Nora blamed themselves for the spill.,reflexive_number_match,prompt_3,openai
3,openai_prompt_3_7a34d523da,anaphor_agreement,family_home,The twins taught themselves to bake bread.,The twins taught herself to bake bread.,reflexive_number_match,prompt_3,openai
4,openai_prompt_3_9133cc22a1,anaphor_agreement,family_home,Grandpa shaved himself before the call.,Grandpa shaved themselves before the call.,reflexive_number_match,prompt_3,openai


## Required Columns

In [17]:
required_columns = [
    "id",
    "phenomenon",
    "topic",
    "good",
    "bad",
    "edit_type",
    "prompt_family",
    "parent_llm",
]

missing_columns = [col for col in required_columns if col not in df.columns]
missing_columns

[]

## Official Word Counts for `good` and `bad`

In [18]:
df["good_word_count"] = df["good"].fillna("").map(count_word)
df["bad_word_count"] = df["bad"].fillna("").map(count_word)
df["pair_word_count"] = df["good_word_count"] + df["bad_word_count"]
df[["good", "bad", "good_word_count", "bad_word_count", "pair_word_count"]].head()

,good,bad,good_word_count,bad_word_count,pair_word_count
0,Leah reminded herself to lock the gate.,Leah reminded themselves to lock the gate.,8,8,16
1,The brothers dressed themselves before dinner.,The brothers dressed himself before dinner.,7,7,14
2,Aunt Nora blamed herself for the spill.,Aunt Nora blamed themselves for the spill.,8,8,16
3,The twins taught themselves to bake bread.,The twins taught herself to bake bread.,8,8,16
4,Grandpa shaved himself before the call.,Grandpa shaved themselves before the call.,7,7,14


In [19]:
word_count_summary = {
    "num_records": int(len(df)),
    "total_good_words": int(df["good_word_count"].sum()),
    "total_bad_words": int(df["bad_word_count"].sum()),
    "total_pair_words": int(df["pair_word_count"].sum()),
    "avg_good_words": float(df["good_word_count"].mean()) if len(df) else 0.0,
    "avg_bad_words": float(df["bad_word_count"].mean()) if len(df) else 0.0,
    "avg_pair_words": float(df["pair_word_count"].mean()) if len(df) else 0.0,
    "min_good_words": int(df["good_word_count"].min()) if len(df) else 0,
    "max_good_words": int(df["good_word_count"].max()) if len(df) else 0,
    "min_bad_words": int(df["bad_word_count"].min()) if len(df) else 0,
    "max_bad_words": int(df["bad_word_count"].max()) if len(df) else 0,
}
word_count_summary

{'num_records': 48393,
 'total_good_words': 433369,
 'total_bad_words': 426881,
 'total_pair_words': 860250,
 'avg_good_words': 8.955200132250532,
 'avg_bad_words': 8.821131155332383,
 'avg_pair_words': 17.776331287582916,
 'min_good_words': 3,
 'max_good_words': 16,
 'min_bad_words': 3,
 'max_bad_words': 17}

## Dataset Overview

In [20]:
overview = {
    "num_records": int(len(df)),
    "num_unique_ids": int(df["id"].nunique(dropna=True)),
    "num_unique_good": int(df["good"].nunique(dropna=True)),
    "num_unique_bad": int(df["bad"].nunique(dropna=True)),
    "num_unique_phenomena": int(df["phenomenon"].nunique(dropna=True)),
    "num_unique_topics": int(df["topic"].nunique(dropna=True)),
    "num_unique_edit_types": int(df["edit_type"].nunique(dropna=True)),
    "num_unique_prompt_families": int(df["prompt_family"].nunique(dropna=True)),
    "num_unique_parent_llms": int(df["parent_llm"].nunique(dropna=True)),
}
overview

{'num_records': 48393,
 'num_unique_ids': 38430,
 'num_unique_good': 35649,
 'num_unique_bad': 36986,
 'num_unique_phenomena': 13,
 'num_unique_topics': 10,
 'num_unique_edit_types': 34,
 'num_unique_prompt_families': 1,
 'num_unique_parent_llms': 3}

In [21]:
df.isna().sum().sort_values(ascending=False)

id                 0
phenomenon         0
topic              0
good               0
bad                0
edit_type          0
prompt_family      0
parent_llm         0
good_word_count    0
bad_word_count     0
pair_word_count    0
dtype: int64

## Counts by Phenomenon

In [22]:
df["phenomenon"].value_counts().rename_axis("phenomenon").reset_index(name="count")

,phenomenon,count
0,island_effects,4041
1,binding,4032
2,control_raising,4032
3,determiner_noun_agreement,4032
4,anaphor_agreement,4032
5,npi_licensing,4032
6,ellipsis,4032
7,filler_gap,4032
8,irregular_forms,4032
9,subject_verb_agreement,4032


## Counts by Topic

In [23]:
df["topic"].value_counts().rename_axis("topic").reset_index(name="count")

,topic,count
0,school_classroom,7893
1,family_home,6992
2,books_stories,5324
3,shopping_market,5211
4,household_objects,4457
5,travel_transport,4124
6,chores_routines,4112
7,park_outdoors,3996
8,food_meals,3682
9,animals_pets,2602


## Counts by Parent LLM

In [24]:
df["parent_llm"].value_counts().rename_axis("parent_llm").reset_index(name="count")

,parent_llm,count
0,grok,16137
1,openai,16128
2,gemini,16128


## Counts by Prompt Family

In [25]:
df["prompt_family"].value_counts().rename_axis("prompt_family").reset_index(name="count")

,prompt_family,count
0,prompt_3,48393


## Counts by Edit Type

In [26]:
df["edit_type"].value_counts().rename_axis("edit_type").reset_index(name="count")

,edit_type,count
0,number_agreement,4032
1,determiner_number_match,4032
2,resumptive_pronoun,3656
3,existential_there_with_strong_quantifier,2298
4,reflexive_number_match,2223
5,one_anaphora_modifier_position,2084
6,ones_anaphora_modifier_position,1948
7,irregular_participle,1904
8,existential_there_quantifier,1734
9,missing_object,1619


## Cross Tabs

In [27]:
pd.crosstab(df["phenomenon"], df["parent_llm"])

parent_llm,gemini,grok,openai
phenomenon,,,
anaphor_agreement,1344,1344,1344
argument_structure,1343,1344,1344
binding,1344,1344,1344
control_raising,1344,1344,1344
determiner_noun_agreement,1344,1344,1344
ellipsis,1344,1344,1344
family_home,1,0,0
filler_gap,1344,1344,1344
irregular_forms,1344,1344,1344


In [28]:
pd.crosstab(df["phenomenon"], df["prompt_family"])

prompt_family,prompt_3
phenomenon,
anaphor_agreement,4032
argument_structure,4031
binding,4032
control_raising,4032
determiner_noun_agreement,4032
ellipsis,4032
family_home,1
filler_gap,4032
irregular_forms,4032


In [29]:
pd.crosstab(df["topic"], df["parent_llm"])

parent_llm,gemini,grok,openai
topic,,,
animals_pets,842,849,911
books_stories,1762,1771,1791
chores_routines,1361,1362,1389
family_home,2285,2340,2367
food_meals,1227,1311,1144
household_objects,1531,1425,1501
park_outdoors,1342,1376,1278
school_classroom,2685,2654,2554
shopping_market,1726,1690,1795


## Length Analysis

In [30]:
df[["good_word_count", "bad_word_count", "pair_word_count"]].describe()

,good_word_count,bad_word_count,pair_word_count
count,48393.000000,48393.000000,48393.000000
mean,8.955200,8.821131,17.776331
std,1.537397,1.961944,3.417047
min,3.000000,3.000000,6.000000
25%,8.000000,8.000000,16.000000
50%,9.000000,9.000000,18.000000
75%,10.000000,10.000000,20.000000
max,16.000000,17.000000,32.000000


In [31]:
df["word_count_gap"] = (df["good_word_count"] - df["bad_word_count"]).abs()
df["word_count_gap"].describe()

count    48393.000000
mean         0.411712
std          0.773233
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          7.000000
Name: word_count_gap, dtype: float64

In [32]:
df.sort_values("pair_word_count", ascending=False)[
    ["id", "phenomenon", "topic", "good", "bad", "good_word_count", "bad_word_count", "pair_word_count"]
].head(20)

,id,phenomenon,topic,good,bad,good_word_count,bad_word_count,pair_word_count
8020,openai_prompt_3_9079690b59,ellipsis,food_meals,"The diner offered the fresh rolls, while the s...","The diner offered the fresh rolls, while the o...",16,16,32
11925,openai_prompt_3_b515193627,island_effects,books_stories,The critic rejected [the draft that included w...,Which twist did the critic reject [the draft t...,15,17,32
11028,openai_prompt_3_b00f91c79f,island_effects,travel_transport,"Which map did the guide fold, and which brochu...","Which map did the guide fold, and which brochu...",15,17,32
11044,openai_prompt_3_0a8bf30c79,island_effects,travel_transport,"Whose luggage did the station manager store, a...","Whose luggage did the station manager store, a...",15,17,32
3377,openai_prompt_3_99d7dc672b,binding,school_classroom,The librarian told the teacher that the pupils...,The librarian told the teacher that the pupils...,16,16,32
275,openai_prompt_3_19fc2ad5bc,anaphor_agreement,school_classroom,"The puppy does not belong in the science room,...","The puppy does not belong in the science room,...",16,16,32
11024,openai_prompt_3_eab34d4323,island_effects,school_classroom,"Which chalk did the teacher pack, and which er...","Which chalk did the teacher pack, and which er...",15,17,32
11158,openai_prompt_3_5259c18ba1,island_effects,books_stories,Which page did the child underline in the chap...,Which page did the child underline in the chap...,14,17,31
11159,openai_prompt_3_8c5a475aea,island_effects,travel_transport,Which station did the report describe in the s...,Which station did the report describe in the s...,14,17,31
27121,grok_prompt_3_e710481441,island_effects,books_stories,The librarian shelved the book that the child ...,Which book did the librarian shelve the one th...,14,17,31


## Quality Checks on Final Training Data

In [33]:
quality_checks = {
    "good_equals_bad": int((df["good"] == df["bad"]).sum()),
    "duplicate_ids": int(df.duplicated(subset=["id"]).sum()),
    "duplicate_training_rows": int(df.duplicated(subset=["phenomenon", "topic", "good", "bad", "edit_type"]).sum()),
    "empty_good": int((df["good"].fillna("").str.strip() == "").sum()),
    "empty_bad": int((df["bad"].fillna("").str.strip() == "").sum()),
}
quality_checks

{'good_equals_bad': 136,
 'duplicate_ids': 9963,
 'duplicate_training_rows': 10166,
 'empty_good': 0,
 'empty_bad': 0}

In [34]:
df[df.duplicated(subset=["phenomenon", "topic", "good", "bad", "edit_type"], keep=False)].sort_values(
    ["phenomenon", "topic", "edit_type"]
)[["id", "phenomenon", "topic", "good", "bad", "edit_type", "prompt_family", "parent_llm"]].head(30)

,id,phenomenon,topic,good,bad,edit_type,prompt_family,parent_llm
262,openai_prompt_3_24b20ebb22,anaphor_agreement,family_home,The baby soothed itself with a soft toy.,The baby soothed himself with a soft toy.,reflexive_animacy_match,prompt_3,openai
396,openai_prompt_3_24b20ebb22,anaphor_agreement,family_home,The baby soothed itself with a soft toy.,The baby soothed himself with a soft toy.,reflexive_animacy_match,prompt_3,openai
469,openai_prompt_3_983a49484b,anaphor_agreement,family_home,The baby covered itself with a blanket.,The baby covered himself with a blanket.,reflexive_animacy_match,prompt_3,openai
885,openai_prompt_3_983a49484b,anaphor_agreement,family_home,The baby covered itself with a blanket.,The baby covered himself with a blanket.,reflexive_animacy_match,prompt_3,openai
1019,openai_prompt_3_64a794034d,anaphor_agreement,family_home,The baby soothed itself with the blanket.,The baby soothed himself with the blanket.,reflexive_animacy_match,prompt_3,openai
1104,openai_prompt_3_64a794034d,anaphor_agreement,family_home,The baby soothed itself with the blanket.,The baby soothed himself with the blanket.,reflexive_animacy_match,prompt_3,openai
16260,grok_prompt_3_cdcc5a196d,anaphor_agreement,family_home,Father blamed himself for the broken window.,Father blamed itself for the broken window.,reflexive_animacy_match,prompt_3,grok
16386,grok_prompt_3_cdcc5a196d,anaphor_agreement,family_home,Father blamed himself for the broken window.,Father blamed itself for the broken window.,reflexive_animacy_match,prompt_3,grok
16502,grok_prompt_3_d8a58b5cb1,anaphor_agreement,family_home,The parents allowed themselves a short break.,The parents allowed itself a short break.,reflexive_animacy_match,prompt_3,grok
16635,grok_prompt_3_9ed2df5e27,anaphor_agreement,family_home,My uncle blamed himself for the broken vase.,My uncle blamed itself for the broken vase.,reflexive_animacy_match,prompt_3,grok


## Samples

In [35]:
df.sample(min(10, len(df)), random_state=42) if len(df) else df

,id,phenomenon,topic,good,bad,edit_type,prompt_family,parent_llm,good_word_count,bad_word_count,pair_word_count,word_count_gap
2514,openai_prompt_3_c97a4cbb96,argument_structure,travel_transport,The captain steered the boat toward shore.,The captain steered toward shore.,missing_object,prompt_3,openai,8,6,14,2
31143,grok_prompt_3_666b69d934,subject_verb_agreement,animals_pets,The birds above the roof sing sweet songs.,The birds above the roof sings sweet songs.,number_agreement,prompt_3,grok,9,9,18,0
26126,grok_prompt_3_df6304ab67,irregular_forms,park_outdoors,The old oak tree was struck by lightning last ...,The old oak tree was striked by lightning last...,irregular_participle,prompt_3,grok,11,11,22,0
27872,grok_prompt_3_76d8d42996,island_effects,school_classroom,Which book did Emma realize that the students ...,Which book did Emma ask whether the students e...,wh_island,prompt_3,grok,10,10,20,0
33430,gemini_prompt_3_128af954d5,anaphor_agreement,school_classroom,The researchers gathered themselves for the me...,The researchers gathered itself for the meeting.,reflexive_number_match,prompt_3,gemini,8,8,16,0
6890,openai_prompt_3_d404b350b6,ellipsis,household_objects,We stored the heavy lamp and returned the ligh...,We stored the heavy lamp and returned the one ...,one_anaphora_modifier_position,prompt_3,openai,11,11,22,0
32999,gemini_prompt_3_230b3913cd,anaphor_agreement,household_objects,The dishwasher drained itself after the cycle.,The dishwasher drained himself after the cycle.,reflexive_animacy_match,prompt_3,gemini,8,8,16,0
40494,gemini_prompt_3_8c9886f189,filler_gap,park_outdoors,What bridge did the engineer inspect for safety?,What bridge did the engineer inspect it for sa...,resumptive_pronoun,prompt_3,gemini,9,10,19,1
3953,openai_prompt_3_c4e9e6f81a,binding,family_home,Ethan heard that Julia would call him after work.,Ethan heard that Julia would call herself afte...,nonlocal_reflexive,prompt_3,openai,10,10,20,0
46723,gemini_prompt_3_f5e22468e2,quantifiers,household_objects,There are several pillows on the bed.,There are all pillows on the bed.,existential_there_with_strong_quantifier,prompt_3,gemini,8,8,16,0


In [36]:
for phenomenon in sorted(df["phenomenon"].dropna().unique()[:5]):
    print(f"\n=== {phenomenon} ===")
    display(df[df["phenomenon"] == phenomenon][["id", "topic", "good", "bad", "edit_type"]].head(3))


=== anaphor_agreement ===


,id,topic,good,bad,edit_type
0,openai_prompt_3_c4e80b6851,family_home,Leah reminded herself to lock the gate.,Leah reminded themselves to lock the gate.,reflexive_number_match
1,openai_prompt_3_71c80d8303,family_home,The brothers dressed themselves before dinner.,The brothers dressed himself before dinner.,reflexive_number_match
2,openai_prompt_3_27aa320f3c,family_home,Aunt Nora blamed herself for the spill.,Aunt Nora blamed themselves for the spill.,reflexive_number_match



=== argument_structure ===


,id,topic,good,bad,edit_type
1344,openai_prompt_3_b5ff515196,chores_routines,Maya folded the towels after breakfast.,Maya folded after breakfast.,missing_object
1345,openai_prompt_3_5b57ab3b1f,family_home,The cat hid under the sofa.,The cat hid the sofa.,illicit_object
1346,openai_prompt_3_92b8719510,park_outdoors,Evan pointed toward the fountain.,Evan pointed.,missing_required_complement



=== binding ===


,id,topic,good,bad,edit_type
2688,openai_prompt_3_3435901689,family_home,The sisters told Mom that Ava trusted them wit...,The sisters told Mom that Ava trusted themselv...,unlicensed_reflexive
2689,openai_prompt_3_87da050bf1,family_home,Liam said that he would wash the dishes after ...,Liam said that himself would wash the dishes a...,reflexive_subject
2690,openai_prompt_3_9f583d275d,family_home,Nora knew that Ben would call her before lunch.,Nora knew that Ben would call herself before l...,nonlocal_reflexive



=== control_raising ===


,id,topic,good,bad,edit_type
4032,openai_prompt_3_b3e15474e8,travel_transport,There appears to be a delay at the station.,There decides to be a delay at the station.,expletive_there_with_raising
4033,openai_prompt_3_9becc134f9,travel_transport,There seems to be room for one more suitcase.,There wants to be room for one more suitcase.,expletive_subject_selection
4034,openai_prompt_3_c77995f566,travel_transport,The route was easy to remember.,The route was willing to remember.,tough_vs_control



=== determiner_noun_agreement ===


,id,topic,good,bad,edit_type
5376,openai_prompt_3_c1797ddd0a,shopping_market,These apples look sweeter than the ones next t...,This apples look sweeter than the ones next to...,determiner_number_match
5377,openai_prompt_3_140267145f,shopping_market,That melon has a deep green rind.,Those melon has a deep green rind.,determiner_number_match
5378,openai_prompt_3_940236141a,shopping_market,Those cherries were cheaper at the corner stall.,That cherries were cheaper at the corner stall.,determiner_number_match


## Optional Export

In [37]:
EXPORT_CSV = False

if EXPORT_CSV:
    df.to_csv("train_ready_grammar_data_eda_export.csv", index=False)
    print("Saved train_ready_grammar_data_eda_export.csv")